# Restaurant Annual Turnover Prediction — Improved Submission

**Student register number:** submission

**Student name:** submission

**Team name:** submission

Predict annual turnover in INR for 500 restaurants. Lower **RMSE** is better.

This notebook contains executable code, saved outputs, findings, model comparisons,
and submission checks. It compares 38 settings and blends the useful models using
predictions made for rows held out from each training fold.

**Google Colab:** upload this notebook, select a CPU runtime, and run all cells.
Upload the four supplied CSV files when prompted. The final cell downloads
`submission.csv`. Download the executed notebook through
Colab's File menu before submitting it to the assignment. Run time depends on the CPU.

## 1. Objective and evaluation

This is a regression problem. The generic classification/accuracy FAQ does not
describe this target. We score **sqrt(mean((actual - predicted)^2))** in original INR.

The previous submitted CSV received **12,204,615.6438253 RMSE, rank 3**, in the user's
11 October 2026 screenshot. The earlier leader scored **11,946,659.0702364**. Those
are portal-reported scores. The new CSV's hidden-test score is unknown until uploaded.

Development uses shuffled five-fold validation (seed 42) to compare settings and
choose weights. Three further shuffled assignments and a City GroupKFold stress
test evaluate fixed models and weights. These checks reuse the labeled dataset;
they are robustness estimates, not an independent test set. Seeds 71 and 2026
were also examined during the earlier 30-setting search. Seed 303 was reserved
until the final candidate was fixed. No hidden targets or leaderboard labels
were used to train models or choose weights.

In [ ]:
import importlib.util, subprocess, sys, warnings, json, hashlib, copy
from pathlib import Path

packages = {'numpy':'numpy','pandas':'pandas','matplotlib':'matplotlib',
            'sklearn':'scikit-learn','scipy':'scipy','catboost':'catboost',
            'lightgbm':'lightgbm','xgboost':'xgboost'}
missing = [package for module,package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, catboost, lightgbm, xgboost
from catboost import CatBoostRegressor
from scipy.optimize import minimize
from sklearn.model_selection import KFold, GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
warnings.filterwarnings('ignore', category=FutureWarning)

SEED = 42
FILE_STEM = 'submission'
OUTPUT_DIR = Path.cwd()
print('Python:',sys.version.split()[0])
print('Versions:', {k:v.__version__ for k,v in {'numpy':np,'pandas':pd,'sklearn':sklearn,'catboost':catboost,'lightgbm':lightgbm,'xgboost':xgboost}.items()})

## 2. Load and audit the inputs

Restaurant registration numbers identify prediction rows. They are different from
the student's register number in the filename and are excluded from model features.
The train/test spelling difference in the endorsement column is aligned explicitly.

In [ ]:
required = ['Train_dataset.csv','Test_dataset.csv','Data_Dictionary.csv','Sample_Solution.csv']
DATA_DIR = Path.cwd()
if not all((DATA_DIR/f).exists() for f in required):
    local = Path('~/Downloads')
    if all((local/f).exists() for f in required):
        DATA_DIR = local
    else:
        try:
            from google.colab import files
        except ImportError:
            raise FileNotFoundError('Place all four CSV files beside this notebook, or set DATA_DIR.')
        files.upload()
        DATA_DIR = Path.cwd()
assert all((DATA_DIR/f).exists() for f in required), 'Upload all four input CSV files.'
train=pd.read_csv(DATA_DIR/required[0]).rename(columns={'Endorsed By':'Endoresed By'})
test=pd.read_csv(DATA_DIR/required[1])
dictionary=pd.read_csv(DATA_DIR/required[2])
sample=pd.read_csv(DATA_DIR/required[3])
y=train['Annual Turnover'].to_numpy(dtype=float)
assert np.isfinite(y).all() and (y>=0).all()
assert 'Annual Turnover' not in test
assert len(test)==500 and test['Registration Number'].is_unique
assert train['Registration Number'].is_unique
assert set(train['Registration Number']).isdisjoint(test['Registration Number'])
assert train.drop(columns='Annual Turnover').columns.tolist()==test.columns.tolist()
assert sample.columns.tolist()==['Registration Number','Annual Turnover']
assert sample['Registration Number'].equals(test['Registration Number'])
print('Train:',train.shape,'Test:',test.shape)
print('All input schema, target, identifier and row-order checks passed.')
print('Dictionary variables:',len(dictionary))
print('Sample predictions are placeholders:',sample['Annual Turnover'].unique().tolist())
input_hashes={f:hashlib.sha256((DATA_DIR/f).read_bytes()).hexdigest() for f in required}
print('Input SHA256 hashes:')
print(json.dumps(input_hashes,indent=2))

## 3. Findings that guide the experiments

Turnover has a long upper tail. A few extreme restaurants contribute much of the
squared error, so we compare standard RMSE training with Huber losses and a
training-only turnover cap. **Every validation target remains unchanged**, including
extreme values. No rows are removed from the RMSE score.

Service and comfort ratings have different distributions in train and test. Their
removal is evaluated as an experiment; their values are not shifted arbitrarily.
Event-rating blanks mean an event is not offered according to the data dictionary,
so the features retain missingness and event availability.

In [ ]:
print(pd.Series(y,name='Annual Turnover').describe(percentiles=[.5,.9,.95,.99]).round(0).to_string())
errors=(y-y.mean())**2
for threshold in [60e6,100e6,200e6]:
    mask=y>threshold
    print(f'Above INR {threshold/1e6:g}m: {mask.sum()} rows; {100*errors[mask].sum()/errors.sum():.1f}% of mean-baseline squared error')
print('Service/comfort train versus test means:')
print(pd.DataFrame({'Train':train[['Service','Comfortablility']].mean(),'Test':test[['Service','Comfortablility']].mean()}).round(3).to_string())
numeric=train.select_dtypes('number').drop(columns='Registration Number')
print('Largest absolute rank correlations (associations, not causation):')
print(numeric.corr(method='spearman')['Annual Turnover'].drop('Annual Turnover').sort_values(key=abs,ascending=False).head(8).round(3).to_string())
fig,ax=plt.subplots(1,2,figsize=(11,3.6))
ax[0].hist(y/1e6,bins=45,color='#2563eb',edgecolor='white')
ax[0].set(xlabel='Annual turnover (INR millions)',ylabel='Restaurants',title='Full target distribution',yscale='log')
ax[1].hist(y[y<=100e6]/1e6,bins=30,color='#0f766e',edgecolor='white')
ax[1].set(xlabel='Annual turnover (INR millions)',ylabel='Restaurants',title='Target at or below INR 100m (EDA only)')
fig.tight_layout(); plt.show()

## 4. Features and model functions

The base features include normalized categories, opening year/month, cuisine parts,
and event indicators. Enhanced variants add opening-date details, social means and
differences, missingness counts, and rating interactions. One variant combines a
small explicit set of city aliases. All feature engineering is deterministic per row.

CatBoost handles categories directly. Other algorithms use imputation and one-hot
encoding fitted **within each training fold**, with unseen categories handled safely.
The spline model adds smooth numeric effects; SVR supplies a different model shape.

The monotonic CatBoost candidate assumes higher social popularity, hygiene and Zomato
ratings should not decrease predicted turnover, and a higher tier number should
not increase it when other features are fixed. This is a tested modeling assumption.
Its feature importance is predictive association, not evidence of causation.

In [ ]:
TARGET = 'Annual Turnover'

ID = 'Registration Number'

EVENTS = ['Live Music Rating','Comedy Gigs Rating','Value Deals Rating','Live Sports Rating']

SOCIAL = ['Facebook Popularity Quotient','Instagram Popularity Quotient']

CORE = SOCIAL + ['Hygiene Rating','Restaurant Zomato Rating','Live Music Rating','Value for Money','Overall Restaurant Rating','Restaurant City Tier','Cuisine','Restaurant Theme','Restaurant Location','Endoresed By','City','Opening Year']

SPECS = {
 'raw_baseline':dict(kind='cat',variant='base',depth=6,iterations=302,loss='RMSE'),
 'huber_baseline':dict(kind='cat',variant='base',depth=6,iterations=768,loss='Huber:delta=12000000'),
 'drop_shift':dict(kind='cat',variant='drop_shift',depth=6),
 'engineered':dict(kind='cat',variant='engineered',depth=6),
 'engineered_drop':dict(kind='cat',variant='engineered_drop',depth=6),
 'core_depth4':dict(kind='cat',variant='core',depth=4),
 'core_depth3':dict(kind='cat',variant='core',depth=3),
 'shallow_depth3':dict(kind='cat',variant='base',depth=3,l2=15),
 'ordered_depth5':dict(kind='cat',variant='base',depth=5,boosting_type='Ordered'),
 'huber20_engineered':dict(kind='cat',variant='engineered_drop',depth=5,loss='Huber:delta=20000000'),
 'huber40':dict(kind='cat',variant='base',depth=5,loss='Huber:delta=40000000'),
 'clip80_engineered':dict(kind='cat',variant='engineered_drop',depth=5,cap=80e6),
 'clip120':dict(kind='cat',variant='base',depth=5,cap=120e6),
 'ridge':dict(kind='ridge',variant='base',alpha=100),
 'spline_ridge':dict(kind='spline',variant='base',alpha=100),
 'spline_core':dict(kind='spline',variant='core',alpha=30),
 'svr_core':dict(kind='svr',variant='core',C=20),
 'hist_core':dict(kind='hist',variant='core'),
 'ratings_depth4':dict(kind='cat',variant='ratings',depth=4),
 'ratings_huber40':dict(kind='cat',variant='ratings',depth=5,loss='Huber:delta=40000000'),
 'ratings_clip100':dict(kind='cat',variant='ratings',depth=5,cap=100e6),
 'no_city_date':dict(kind='cat',variant='no_city_date',depth=5),
 'no_city_huber40':dict(kind='cat',variant='no_city_date',depth=5,loss='Huber:delta=40000000'),
 'clip120_depth6':dict(kind='cat',variant='base',depth=6,cap=120e6),
 'clip120_depth4':dict(kind='cat',variant='base',depth=4,cap=120e6),
 'huber40_depth6':dict(kind='cat',variant='base',depth=6,loss='Huber:delta=40000000'),
 'huber80_depth5':dict(kind='cat',variant='base',depth=5,loss='Huber:delta=80000000'),
 'huber40_engineered':dict(kind='cat',variant='engineered',depth=5,loss='Huber:delta=40000000'),
 'huber40_monotone':dict(kind='cat',variant='base',depth=5,loss='Huber:delta=40000000',monotone=True),
 'extra_clip100':dict(kind='extra',variant='base',cap=100e6),
 'lgb15':dict(kind='lgb',variant='base',leaves=15,leaf_samples=60),
 'lgb7':dict(kind='lgb',variant='base',leaves=7,leaf_samples=80),
 'lgb_huber':dict(kind='lgb',variant='base',leaves=15,leaf_samples=60,objective='huber'),
 'lgb_clip120':dict(kind='lgb',variant='base',leaves=15,leaf_samples=60,cap=120e6),
 'xgb_depth3':dict(kind='xgb',variant='base',depth=3),
 'xgb_depth5':dict(kind='xgb',variant='base',depth=5),
 'xgb_huber':dict(kind='xgb',variant='base',depth=3,objective='reg:pseudohubererror'),
 'xgb_clip120':dict(kind='xgb',variant='base',depth=3,cap=120e6),
}

def features(raw, variant='base'):
    data=raw.drop(columns=[TARGET,ID],errors='ignore').copy()
    dates=pd.to_datetime(data.pop('Opening Day of Restaurant'),format='%d-%m-%Y',errors='coerce')
    data['Opening Year']=dates.dt.year.astype(float)
    data['Opening Month']=dates.dt.month.astype(float)
    for c in EVENTS:
        data['Offers '+c.replace(' Rating','')]=data[c].notna().astype(int)
    parts=data['Cuisine'].astype('string').fillna('Missing').str.split(',',n=1,expand=True)
    data['Cuisine 1']=parts[0].str.strip().str.lower()
    data['Cuisine 2']=parts.reindex(columns=[0,1])[1].astype('string').str.strip().str.lower()
    for c in ['Cuisine','City','Restaurant Location','Endoresed By','Restaurant Type','Restaurant Theme']:
        data[c]=data[c].astype('string').str.strip().str.lower().str.replace(r'\s+',' ',regex=True).fillna('Missing')
    data['City Is Unknown']=(data['City']=='-1').astype(int)
    for c in data.select_dtypes(include=['object','string']).columns:
        data[c]=data[c].fillna('Missing').astype(str)
    if variant in ['engineered','engineered_drop']:
        data['City']=data['City'].replace({'bengaluru':'bangalore','gurugram':'gurgaon','bombay':'mumbai','calcutta':'kolkata','delhi':'new delhi'})
        data['Cuisine unordered']=['|'.join(sorted(v.split(','))) for v in data['Cuisine']]
        data['Opening Day']=dates.dt.day.astype(float)
        data['Opening Days Since 1970']=(dates-pd.Timestamp('1970-01-01')).dt.days.astype(float)
        data['Social mean']=data[SOCIAL].mean(axis=1)
        data['Social difference']=data[SOCIAL[0]]-data[SOCIAL[1]]
        data['Social missing count']=data[SOCIAL].isna().sum(axis=1)
        data['Rating missing count']=data.isna().sum(axis=1)
        data['Event count']=data[EVENTS].notna().sum(axis=1)
        data['Event mean']=data[EVENTS].mean(axis=1)
        data['Quality mean']=data[['Food Rating','Hygiene Rating','Overall Restaurant Rating']].mean(axis=1)
        data['Social x Zomato']=data['Social mean']*data['Restaurant Zomato Rating']
        data['Social x Hygiene']=data['Social mean']*data['Hygiene Rating']
        data['Quality x Value']=data['Quality mean']*data['Value for Money']
    if variant in ['drop_shift','engineered_drop']:
        data=data.drop(columns=['Service','Comfortablility'])
    if variant=='core': data=data[CORE]
    if variant=='ratings':
        keep=SOCIAL+['Resturant Tier','Restaurant Zomato Rating','Restaurant City Tier','Staff Responsivness','Value for Money','Hygiene Rating','Overall Restaurant Rating']+EVENTS+['Lively','Ambience','Privacy']
        data=data[keep]
    if variant=='no_city_date':
        data=data.drop(columns=['City','City Is Unknown','Opening Year','Opening Month'])
    return data

def rmse(y,p): return float(np.sqrt(np.mean(np.square(np.asarray(y)-np.asarray(p)))))

def sklearn_model(X,spec,seed):
    cats=X.select_dtypes(include=['object','string']).columns.tolist()
    nums=[c for c in X if c not in cats]
    num=make_pipeline(SimpleImputer(strategy='median',add_indicator=True),StandardScaler())
    if spec['kind']=='spline':
        num=make_pipeline(SimpleImputer(strategy='median',add_indicator=True),SplineTransformer(n_knots=4,degree=2,include_bias=False),StandardScaler())
    pre=ColumnTransformer([('num',num,nums),('cat',OneHotEncoder(handle_unknown='ignore',min_frequency=10,sparse_output=False),cats)])
    kind=spec['kind']
    if kind in ['ridge','spline']: model=Ridge(alpha=spec.get('alpha',100))
    elif kind=='svr': model=SVR(C=spec.get('C',20),epsilon=.5,kernel='rbf')
    elif kind=='hist': model=HistGradientBoostingRegressor(max_iter=300,max_leaf_nodes=10,max_depth=4,learning_rate=.035,l2_regularization=30,min_samples_leaf=60,random_state=seed)
    elif kind=='extra': model=ExtraTreesRegressor(n_estimators=400,min_samples_leaf=5,max_features=.7,n_jobs=4,random_state=seed)
    return make_pipeline(pre,model)

def fit_predict(X,y,train_idx,valid_idx,spec,seed,fixed_iterations=None,return_fitted=False):
    ys=np.asarray(y)[train_idx].copy()
    if 'cap' in spec: ys=np.minimum(ys,spec['cap'])
    if spec['kind']=='cat':
        params=dict(loss_function=spec.get('loss','RMSE'),eval_metric='RMSE',iterations=fixed_iterations or spec.get('iterations',1600),learning_rate=.04,depth=spec.get('depth',6),l2_leaf_reg=spec.get('l2',5),random_seed=seed,thread_count=4,verbose=False,allow_writing_files=False)
        if 'boosting_type' in spec: params['boosting_type']=spec['boosting_type']
        if spec.get('monotone'): params['monotone_constraints']={**{c:1 for c in SOCIAL+['Hygiene Rating','Restaurant Zomato Rating']},'Resturant Tier':-1}
        model=CatBoostRegressor(**params)
        cat=X.select_dtypes(include=['object','string']).columns.tolist()
        stopping={} if fixed_iterations or 'iterations' in spec else dict(eval_set=(X.iloc[valid_idx],np.asarray(y)[valid_idx]),early_stopping_rounds=120)
        model.fit(X.iloc[train_idx],ys,cat_features=cat,**stopping)
        p=model.predict(X.iloc[valid_idx]); trees=model.tree_count_
    elif spec['kind'] in ['lgb','xgb']:
        pre=sklearn_model(X,dict(kind='ridge'),seed)[:-1]
        ztr=pre.fit_transform(X.iloc[train_idx]); zva=pre.transform(X.iloc[valid_idx])
        early=not (fixed_iterations or 'iterations' in spec)
        iterations=fixed_iterations or spec.get('iterations',1400)
        if spec['kind']=='lgb':
            from lightgbm import LGBMRegressor,early_stopping
            model=LGBMRegressor(n_estimators=iterations,learning_rate=.025,num_leaves=spec.get('leaves',15),min_child_samples=spec.get('leaf_samples',60),reg_lambda=20,reg_alpha=.1,colsample_bytree=.8,subsample=.8,subsample_freq=1,random_state=seed,n_jobs=4,verbosity=-1,objective=spec.get('objective','regression'))
            params=dict(eval_set=[(zva,np.asarray(y)[valid_idx]/1e6)],eval_metric='rmse',callbacks=[early_stopping(100,verbose=False,first_metric_only=True)]) if early else {}
            model.fit(ztr,ys/1e6,**params)
            trees=model.best_iteration_ if early else iterations
        else:
            from xgboost import XGBRegressor
            params=dict(n_estimators=iterations,max_depth=spec.get('depth',3),learning_rate=.025,min_child_weight=30,reg_lambda=20,reg_alpha=.1,colsample_bytree=.8,subsample=.8,random_state=seed,n_jobs=4,tree_method='hist',objective=spec.get('objective','reg:squarederror'),eval_metric='rmse',base_score=float(ys.mean()/1e6))
            if early: params['early_stopping_rounds']=100
            if spec.get('objective')=='reg:pseudohubererror': params['huber_slope']=12
            model=XGBRegressor(**params)
            model.fit(ztr,ys/1e6,**(dict(eval_set=[(zva,np.asarray(y)[valid_idx]/1e6)],verbose=False) if early else {}))
            trees=model.best_iteration+1 if early else iterations
        p=model.predict(zva)*1e6
    else:
        model=sklearn_model(X,spec,seed)
        model.fit(X.iloc[train_idx],ys/1e6)
        p=model.predict(X.iloc[valid_idx])*1e6; trees=0
    if return_fitted:
        fitted=dict(model=model,preprocessor=pre if spec['kind'] in ['lgb','xgb'] else None,scale=1 if spec['kind']=='cat' else 1e6)
        return np.maximum(p,0),trees,fitted
    return np.maximum(p,0),trees

def predict_fitted(fitted,X):
    transformed=fitted['preprocessor'].transform(X) if fitted['preprocessor'] is not None else X
    return np.maximum(fitted['model'].predict(transformed)*fitted['scale'],0)

In [ ]:
feature_data={variant:features(train,variant) for variant in {s['variant'] for s in SPECS.values()}}
for variant,X in feature_data.items():
    assert 'Annual Turnover' not in X and 'Registration Number' not in X
    assert X.columns.equals(features(test,variant).columns)
    changed=train.copy(); changed['Annual Turnover']=0; changed['Registration Number']=-1
    pd.testing.assert_frame_equal(X,features(changed,variant))
assert np.isclose(rmse([0,3],[0,0]),np.sqrt(4.5))
print('RMSE formula check, target/ID exclusion checks and train/test feature alignment passed.')
print('Base feature count:',feature_data['base'].shape[1])

## 5. Development comparison: 38 settings

Settings include CatBoost, LightGBM, XGBoost, Extra Trees, histogram boosting, Ridge,
spline Ridge and SVR. Early stopping and model selection use these development
folds, so their scores are selection estimates. The later checks use fixed tree
counts and fixed ensemble weights. The rank-3 baseline is reproduced as
0.55 × raw CatBoost (302 trees) + 0.45 × Huber CatBoost (768 trees).

In [ ]:
dev_folds=list(KFold(5,shuffle=True,random_state=42).split(train))
coverage=np.zeros(len(train),dtype=int)
for tr,va in dev_folds:
    assert set(tr).isdisjoint(va)
    coverage[va]+=1
assert (coverage==1).all()
dev_predictions={}; dev_trees={}; dev_rows=[]
for name,spec in SPECS.items():
    X=feature_data[spec['variant']]; pred=np.zeros(len(y)); trees=[]
    for f,(tr,va) in enumerate(dev_folds,1):
        pred[va],n=fit_predict(X,y,tr,va,spec,42+f)
        trees.append(n)
    dev_predictions[name]=pred; dev_trees[name]=trees
    dev_rows.append({'Model':name,'Development RMSE':rmse(y,pred)})
    print(f'{name:25s} RMSE {rmse(y,pred):,.0f}')
development_table=pd.DataFrame(dev_rows).sort_values('Development RMSE')
old_dev=.55*dev_predictions['raw_baseline']+.45*dev_predictions['huber_baseline']
print(f'Rank-3 model, fixed configuration on development folds: {rmse(y,old_dev):,.0f}')
print(development_table.head(10).to_string(index=False,float_format=lambda v:f'{v:,.0f}'))

## 6. Freeze the ensemble before further checks

Fit nonnegative weights that sum to one using development out-of-fold predictions.
The objective is squared error in INR millions plus `2 × sum(weight²)`, which
discourages placing all weight on one model. Drop weights below 5% and renormalize.
Tree counts are the median selected across development folds. No later validation
score is used to adjust the weights. The published test score from the old model
is a reference only and does not enter this calculation.

In [ ]:
names=list(SPECS)
matrix=np.column_stack([dev_predictions[n] for n in names])/1e6
opt=minimize(lambda w:np.mean((matrix@w-y/1e6)**2)+2*np.sum(w*w),np.ones(len(names))/len(names),
             bounds=[(0,1)]*len(names),constraints=[{'type':'eq','fun':lambda w:w.sum()-1}],
             method='SLSQP',options={'maxiter':1000,'ftol':1e-10})
assert opt.success,opt.message
weights=np.where(opt.x>=.05,opt.x,0); weights/=weights.sum()
ensemble_weights={n:float(w) for n,w in zip(names,weights) if w>0}
fixed_specs={}
for name in ensemble_weights:
    spec=copy.deepcopy(SPECS[name])
    if spec['kind'] in ['cat','lgb','xgb']: spec['iterations']=int(np.median(dev_trees[name]))
    fixed_specs[name]=spec
new_dev=sum(w*dev_predictions[n] for n,w in ensemble_weights.items())
print(pd.Series(ensemble_weights,name='Weight').sort_values(ascending=False).round(4).to_string())
print(f'Development ensemble RMSE: {rmse(y,new_dev):,.0f}')

## 7. Repeated validation and an unseen-city stress test

Compare both models on identical folds, without using the validation targets for
early stopping. Use three shuffled five-fold assignments and five city groups.
City grouping combines explicit aliases (such as Delhi/New Delhi) and prevents a city's rows from appearing in both training and validation
within a fold, making it a harder check for restaurants in unseen cities.

These are repeat checks on the same labeled dataset. They cannot establish the new
hidden-test RMSE, and agreement across them cannot guarantee first place.

In [ ]:
validation_schemes={f'Shuffled 5-fold seed {seed}':list(KFold(5,shuffle=True,random_state=seed).split(train)) for seed in [71,2026,303]}
city_groups=feature_data['engineered']['City']  # Group explicit aliases together.
validation_schemes['City GroupKFold']=list(GroupKFold(5).split(train,y,city_groups))
validation_specs={'raw_baseline':SPECS['raw_baseline'],'huber_baseline':SPECS['huber_baseline'],**fixed_specs}
validation_rows=[]; validation_predictions={}
for label,folds in validation_schemes.items():
    pp={}
    coverage=np.zeros(len(y),dtype=int)
    for tr,va in folds:
        assert set(tr).isdisjoint(va)
        if label=='City GroupKFold': assert set(city_groups.iloc[tr]).isdisjoint(city_groups.iloc[va])
        coverage[va]+=1
    assert (coverage==1).all()
    for name,spec in validation_specs.items():
        pred=np.zeros(len(y)); X=feature_data[spec['variant']]
        for f,(tr,va) in enumerate(folds,1):
            pred[va],_=fit_predict(X,y,tr,va,spec,42+f,spec.get('iterations'))
        pp[name]=pred
    old=.55*pp['raw_baseline']+.45*pp['huber_baseline']
    new=sum(w*pp[n] for n,w in ensemble_weights.items())
    validation_predictions[label]={'old':old,'new':new}
    row={'Validation':label,'Rank-3 model RMSE':rmse(y,old),'New candidate RMSE':rmse(y,new),'Reduction':rmse(y,old)-rmse(y,new)}
    validation_rows.append(row)
    print(f'{label}: old {row["Rank-3 model RMSE"]:,.0f}; new {row["New candidate RMSE"]:,.0f}; reduction {row["Reduction"]:,.0f}')
validation_table=pd.DataFrame(validation_rows)
random_table=validation_table[validation_table['Validation'].str.startswith('Shuffled')]
print('Mean across three shuffled assignments:')
print(random_table.drop(columns='Validation').mean().round(0).to_string())

## 8. Interpret the improvement and its uncertainty

The full-data RMSE above is the primary measure. The extra diagnostic for turnover
at or below INR 100 million helps explain sensitivity to the extreme tail; it is
not a replacement score and does not exclude those rows from model selection.

The bootstrap below resamples restaurant rows from the repeated validation errors.
Its interval describes sampling uncertainty conditional on the fitted models;
it does not cover every source of uncertainty, including future distribution changes.

In [ ]:
random_predictions=[v for k,v in validation_predictions.items() if k.startswith('Shuffled')]
old_sq=np.mean([(y-v['old'])**2 for v in random_predictions],axis=0)
new_sq=np.mean([(y-v['new'])**2 for v in random_predictions],axis=0)
rng=np.random.default_rng(412)
gains=[]
for _ in range(1000):
    ix=rng.integers(0,len(y),len(y))
    gains.append(np.sqrt(old_sq[ix].mean())-np.sqrt(new_sq[ix].mean()))
gain_ci=np.quantile(gains,[.025,.975])
print(f'Conditional 95% bootstrap interval for RMSE reduction: [{gain_ci[0]:,.0f}, {gain_ci[1]:,.0f}]')
body=y<=100e6
print(f'Diagnostic RMSE for targets <= INR 100m: old {np.sqrt(old_sq[body].mean()):,.0f}; new {np.sqrt(new_sq[body].mean()):,.0f}')
print('All validation rows were included in the primary results.')
print('Previous portal-reported RMSE: 12,204,615.64; new portal score: not yet available.')
fig,ax=plt.subplots(figsize=(10,4))
x=np.arange(len(validation_table)); width=.36
ax.bar(x-width/2,validation_table['Rank-3 model RMSE']/1e6,width,label='Rank-3 model',color='#94a3b8')
ax.bar(x+width/2,validation_table['New candidate RMSE']/1e6,width,label='New candidate',color='#2563eb')
ax.set_xticks(x,['Shuffle 71','Shuffle 2026','Shuffle 303','Unseen cities'])
ax.set_ylabel('RMSE (INR millions)'); ax.set_title('Validation with fixed model settings; lower is better')
ax.legend(); fig.tight_layout(); plt.show()

## 9. Train on all labeled data and predict the test restaurants

Each selected model is refitted on all training rows using its frozen settings.
The final prediction is the weighted average. Nonnegative predictions are enforced.
Test targets are unavailable and are never imputed or used for fitting.

In [ ]:
full_predictions={}; fitted_models={}; test_features={}
tr_idx=np.arange(len(train)); te_idx=np.arange(len(train),len(train)+len(test))
combined_target=np.r_[y,np.full(len(test),np.nan)]
for name,spec in fixed_specs.items():
    variant=spec['variant']; tx=features(test,variant)
    both=pd.concat([feature_data[variant],tx],ignore_index=True)
    pred,trees,fitted=fit_predict(both,combined_target,tr_idx,te_idx,spec,42,spec.get('iterations'),return_fitted=True)
    full_predictions[name]=pred; fitted_models[name]=fitted; test_features[name]=tx
    print(f'Fitted {name}; trees={trees}; weight={ensemble_weights[name]:.4f}')
test_prediction=sum(w*full_predictions[n] for n,w in ensemble_weights.items())

# Reproduce the old final model for an auditable comparison of prediction files.
old_parts=[]
for name in ['raw_baseline','huber_baseline']:
    spec=SPECS[name]; both=pd.concat([feature_data['base'],features(test)],ignore_index=True)
    p,_=fit_predict(both,combined_target,tr_idx,te_idx,spec,42,spec['iterations'])
    old_parts.append(p)
old_test_prediction=.55*old_parts[0]+.45*old_parts[1]
archived=Path('rank3_backup')/(FILE_STEM+'.csv')
if archived.exists():
    archived_df=pd.read_csv(archived)
    assert archived_df['Registration Number'].equals(test['Registration Number'])
    assert np.allclose(old_test_prediction,archived_df['Annual Turnover'],rtol=1e-10,atol=.01)
    print('Old model reproduces the archived rank-3 CSV within INR 0.01 tolerance.')
print(f'Mean absolute prediction change: INR {np.mean(np.abs(test_prediction-old_test_prediction)):,.0f}')
importance_name=next(n for n in ensemble_weights if fixed_specs[n]['kind']=='cat')
importance_model=fitted_models[importance_name]['model']
importance=pd.Series(importance_model.feature_importances_,index=importance_model.feature_names_).sort_values(ascending=False)
print('Predictive feature importance for',importance_name,':')
print(importance.head(10).round(2).to_string())

## 10. Check prediction behavior and export the required format

Checks cover ID/target leakage in feature preparation, disjoint validation folds,
unseen categorical values, missing values, row permutation, and CSV round-trip.
They cannot check prediction accuracy on the test rows without the hidden targets.

In [ ]:
# Predictions must follow restaurant rows even after their order is changed.
perm=np.random.default_rng(82).permutation(len(test))
permuted=sum(ensemble_weights[n]*predict_fitted(fitted_models[n],features(test.iloc[perm],fixed_specs[n]['variant'])) for n in ensemble_weights)
assert np.allclose(permuted,test_prediction[perm],rtol=1e-10,atol=.01)

# Small edge case: an unseen city, missing cuisine and missing numeric ratings.
edge=test.iloc[:2].copy()
edge['City']='new_unseen_city_for_validation'
edge['Cuisine']=pd.NA
edge['Facebook Popularity Quotient']=np.nan
edge['Live Music Rating']=np.nan
edge_prediction=sum(ensemble_weights[n]*predict_fitted(fitted_models[n],features(edge,fixed_specs[n]['variant'])) for n in ensemble_weights)
assert np.isfinite(edge_prediction).all() and (edge_prediction>=0).all()

submission=pd.DataFrame({'Registration Number':test['Registration Number'].to_numpy(),'Annual Turnover':test_prediction})
assert submission.shape==(500,2)
assert submission.columns.tolist()==['Registration Number','Annual Turnover']
assert submission['Registration Number'].equals(sample['Registration Number'])
assert submission['Registration Number'].is_unique
assert np.isfinite(submission['Annual Turnover']).all() and (submission['Annual Turnover']>=0).all()
output_path=OUTPUT_DIR/(FILE_STEM+'.csv')
submission.to_csv(output_path,index=False)
reloaded=pd.read_csv(output_path)
assert reloaded.shape==(500,2)
assert reloaded['Registration Number'].equals(test['Registration Number'])
assert np.allclose(reloaded['Annual Turnover'],test_prediction,rtol=1e-12,atol=.001)
assert len(output_path.read_text().splitlines())==501
print('PASS: row-order invariance, unseen/missing feature cases, finite nonnegative predictions, IDs, headers, 500 rows and CSV round-trip.')
print('Saved:',output_path.name)
print('SHA256:',hashlib.sha256(output_path.read_bytes()).hexdigest())
print('Prediction range (INR):',round(test_prediction.min()),'to',round(test_prediction.max()))
print(submission.head().to_string(index=False))

## 11. Findings and submission instructions

1. Extreme turnover values explain much of the training RMSE. Huber training and
   training-only capping help manage their influence; validation retains every target.
2. The selected ensemble combines useful models with different error patterns.
   LightGBM and XGBoost were evaluated but did not receive retained ensemble weight
   in this run. More algorithms do not automatically improve the result.
3. Validation scores and portal scores use the same stated RMSE metric on different
   data. This notebook's validation values are not forecasts of the new portal score.
4. An improvement across repeated splits supports trying the candidate, but the
   gain may be small and the hidden-test ranking can still worsen. The previous
   rank-3 notebook and CSV were retained as a backup.

Upload the generated CSV to **assignment M.Tech. MLA Hackathon -Restaurant Turnover
Prediction**. Upload this executed `.ipynb`, including its outputs, to **LLJ_Oct_2026**
when your model choice is final. The instructions say the assignment allows one
submission, while the hackathon allows multiple CSV submissions. Filenames do not
change the portal registration name; follow the faculty's registration format there.
Review the code and explain these choices in your own words.

In [ ]:
try:
    from google.colab import files
    files.download(str(output_path))
except ImportError:
    print('The prediction CSV is ready in the notebook folder.')